# Granularity
CodeGraphene supports parsing code at different levels of detail: `LINE`, `METHOD`, and `FILE`.
We will point our pipeline at `sample_code.py` and see how changing the granularity alters the graph.

## Setup and Imports

In [1]:
from codegraphene.core import NodeGranularity
from codegraphene.parsers.joern import JoernParser
from codegraphene.trimmers.khop import KHopTrimmer
from codegraphene.serializers.text import CodeReconstructionSerializer
from codegraphene.pipeline import GraphPipeline

target_file = "sample_code.py"

## Line-Level Granularity
Each node represents a single line. We target the query execution (line 63) and look 1 hop away.

In [2]:
pipeline_line = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.LINE),
    trimmer=KHopTrimmer(hops=1),
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.LINE)
)

line_output = pipeline_line.run(target_file, target=63)
print("\n--- FINAL LINE PROMPT ---")
print(line_output)

[Pipeline] Parsing sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpw_2kwyg5/cpg.bin
[JoernParser] Running: joern-export /tmp/tmpw_2kwyg5/cpg.bin --repr all --out /tmp/tmpw_2kwyg5/export
[JoernParser] Ingesting DOT file into NetworkX...
[Pipeline] Trimming graph around 63 (Node 25769803796)...
[Pipeline] Trimmed from 2035 to 7 nodes.
[Pipeline] Serializing subgraph...

--- FINAL LINE PROMPT ---
Line 62: tmp7 = self.scheduler
self.scheduler.step()
Line 63: tmp7 = self.scheduler
self.scheduler.step()
Line 66: tmp9


## Method-Level Granularity
All AST tokens inside a function are collapsed into a single "Method" node. Instead of targeting a line number, we can now target the method by its name!

In [3]:
# With METHOD granularity, the target node must be a string (method name), unlike the line number we passed for LINE granularity.
target_method_name = "step"

pipeline_method = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.METHOD),
    trimmer=KHopTrimmer(hops=2),
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.METHOD)
)

method_output = pipeline_method.run(target_file, target=target_method_name)
print("\n--- FINAL METHOD PROMPT ---")
print(method_output)

[Pipeline] Parsing sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpz5bevbyf/cpg.bin
[JoernParser] Running: joern-export /tmp/tmpz5bevbyf/cpg.bin --repr all --out /tmp/tmpz5bevbyf/export
[JoernParser] Ingesting DOT file into NetworkX...
[Pipeline] Trimming graph around 'step' (Node 107374182403)...
[Pipeline] Trimmed from 171 to 56 nodes.
[Pipeline] Serializing subgraph...

--- FINAL METHOD PROMPT ---
:<module>.get_batch_single_loader
:<module>.recover_states
:<module>.epoch_callback_exec
:<module>.step_after_roll_back
:<module>.synchronize_params
:<module>.configure_distributed_training
:<module>.step_normal
:<module>.set_grads
:<module>.cache_states
:<module>.check_ready
:<module>.step_normal
:<module>.backward
:<module>.clip_grad
:<module>.recover_states
:<module>.step
:<module>.log
:<module>
:<module>.is_implemented
:<module>.check_ready
:<module>.log
:<module>.get_opt_state_for_param
:<module>

## File-Level Granularity
The entire file is collapsed into one node. In a multi-file repository, edges would connect to imported files.

In [7]:
# With FILE granularity, the target node must be a string (path to file relative to target_file).
# In this single-file example, we pass the empty string as the target - it points to target_file.

target_name = ""

pipeline_file = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.FILE),
    trimmer=KHopTrimmer(hops=2),
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.FILE)
)

file_output = pipeline_file.run(target_file, target=target_name)
print("\n--- FINAL FILE PROMPT ---")
print(file_output)

[Pipeline] Parsing sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmp_muqhkn2/cpg.bin
[JoernParser] Running: joern-export /tmp/tmp_muqhkn2/cpg.bin --repr all --out /tmp/tmp_muqhkn2/export
[JoernParser] Ingesting DOT file into NetworkX...
[Pipeline] Trimming graph around '' (Node 21474836480)...
[Pipeline] Trimmed from 1863 to 169 nodes.
[Pipeline] Serializing subgraph...

--- FINAL FILE PROMPT ---
<operator>.fieldAccess
<operator>.assignmentPlus
<operator>.assignment
tmp6
<operator>.assignment
tmp1
<operator>.fieldAccess
tmp2
<module>
<operator>.assignment
self
<operator>.logicalAnd
self
cache_states
self
tmp4
log
tmp2
<operator>.equals
tmp0
<operator>.listLiteral
self
<operator>.fieldAccess
self
self
<operator>.logicalNot
global_step
tmp5
self
<operator>.fieldAccess
<operator>.fieldAccess
self
<operator>.fieldAccess
problem
self
tmp3
loss_dict
<operator>.fieldAccess
<operator>.greaterThan
check_rea